In [ ]:
import pandas as pd
import json
import os
import lxml

In [24]:
df = pd.read_csv(r"data/ner_bioes.csv")

In [25]:
df.head()

,sentence,token,tag
0,Apple launches new iPhone in California,Apple,S-ORG
1,Apple launches new iPhone in California,launches,O
2,Apple launches new iPhone in California,new,O
3,Apple launches new iPhone in California,iPhone,O
4,Apple launches new iPhone in California,in,O


In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   sentence  100 non-null    str  
 1   token     100 non-null    str  
 2   tag       100 non-null    str  
dtypes: str(3)
memory usage: 2.5 KB


In [27]:
df['token'].isna().value_counts()

token
False    100
Name: count, dtype: int64

In [28]:
df['tag'].isna().value_counts()

tag
False    100
Name: count, dtype: int64

In [29]:
df['sentence'].isna().value_counts()

sentence
False    100
Name: count, dtype: int64

In [6]:
res = []
for sentence, group in df.groupby('sentence', sort=False):
    res.append({
        "sentence": sentence,
        "token": group['token'].tolist(),
        "tag": group['tag'].tolist()
    })

In [12]:
# print preview 
for r in res[:3]:
    print(json.dumps(r, indent=2))

{
  "sentence": "Apple launches new iPhone in California",
  "token": [
    "Apple",
    "launches",
    "new",
    "iPhone",
    "in",
    "California"
  ],
  "tag": [
    "S-ORG",
    "O",
    "O",
    "O",
    "O",
    "S-LOC"
  ]
}
{
  "sentence": "Elon Musk founded SpaceX",
  "token": [
    "Elon",
    "Musk",
    "founded",
    "SpaceX"
  ],
  "tag": [
    "B-PER",
    "E-PER",
    "O",
    "S-ORG"
  ]
}
{
  "sentence": "Google opens office in London",
  "token": [
    "Google",
    "opens",
    "office",
    "in",
    "London"
  ],
  "tag": [
    "S-ORG",
    "O",
    "O",
    "O",
    "S-LOC"
  ]
}


In [14]:
# saving

os.makedirs(os.path.dirname("data/ner_group.json"), exist_ok=True)
with open("data/ner_group.json", "w", encoding="utf-8") as f:
    json.dump(res, f, indent=2, ensure_ascii=False)
 
print(f"── {len(res)} sentences saved")

── 22 sentences saved


In [17]:
def parsetag(tag):
    if tag =="0":
        return "0", None
    parts = tag.split(" ",1)
    return (parts[0], parts[1]) if len(parts) ==2 else(tag, None)

In [19]:
def validate_and_fix(tokens,tags):
    tags = list(tags)
    fixed = list(tags)

    warnings = []
    open_type=None
    open_idx = None

    for i, tag in enumerate(tags):
        prefex, etype = parsetag(tag)

        if prefex == "0":
            if open_type is not None:
                warnings.append(f"[idx{open_idx}] B-{open_type}('{tokens[open_idx]}') never closed → S-{open_type}")
                fixed[open_idx] = f"S-{open_type}"
                open_type = open_idx = None
        elif prefex == "S":
            if open_type is not None:
                warnings.append(f"[idx {open_idx}] B-{open_type} ('{tokens[open_idx]}') interrupted by S-{etype} → S-{open_type}")
                fixed[open_idx] = f"S-{open_type}"
                open_type = open_idx = None

        elif prefex == "B":
            if open_type is not None:
                warnings.append(f"[idx {open_idx}] B-{open_type} ('{tokens[open_idx]}') interrupted by new B-{etype} → S-{open_type}")
                fixed[open_idx] = f"S-{open_type}"
            open_type = etype; open_idx = i

        elif prefex == "I":
            if open_type is None:
                warnings.append(f"[idx {i}] I-{etype} ('{tokens[i]}') has no preceding B → S-{etype}")
                fixed[i] = f"S-{etype}"
            elif open_type != etype:
                warnings.append(f"[idx {i}] Type mismatch: open B-{open_type} but I-{etype} ('{tokens[i]}') → I-{open_type}")
                fixed[i] = f"I-{open_type}"

        elif prefex == "E":
            if open_type is None:
                warnings.append(f"[idx {i}] E-{etype} ('{tokens[i]}') has no preceding B → S-{etype}")
                fixed[i] = f"S-{etype}"; open_type = open_idx = None
            elif open_type != etype:
                warnings.append(f"[idx {i}] Type mismatch: open B-{open_type} but E-{etype} ('{tokens[i]}') → E-{open_type}")
                fixed[i] = f"E-{open_type}"; open_type = open_idx = None
            else:
                open_type = open_idx = None

    if open_type is not None:
        warnings.append(f"[idx {open_idx}] B-{open_type} ('{tokens[open_idx]}') never closed at end → S-{open_type}")
        fixed[open_idx] = f"S-{open_type}"

    return fixed, warnings
            

In [20]:
report = []; all_fixed_tags = []; total_warnings = 0

for r in res:
    fixed_tags, warnings = validate_and_fix(r["token"], r["tag"])
    total_warnings += len(warnings)
    all_fixed_tags.extend(fixed_tags)
    report.append({**r, "fixed_tag": fixed_tags, "warnings": warnings})

    status = "✗ ISSUES" if warnings else "✓ OK"
    print(f"[{status}] \"{r['sentence']}\"")
    for w in warnings:
        print(f"  ⚠  {w}")

print(f"\n── {len(report)} sentences | {total_warnings} warnings")

[✓ OK] "Apple launches new iPhone in California"
[✓ OK] "Elon Musk founded SpaceX"
[✓ OK] "Google opens office in London"
[✓ OK] "Microsoft hires engineers in Berlin"
[✓ OK] "Paris is beautiful in summer"
[✓ OK] "Barack Obama visited Germany"
[✓ OK] "Tesla builds factories in Texas"
[✓ OK] "Amazon expands in Europe"
[✓ OK] "Tim Cook speaks at Apple event"
[✓ OK] "London hosts tech conference"
[✓ OK] "Facebook rebrands to Meta"
[✓ OK] "Sundar Pichai leads Google"
[✓ OK] "Berlin becomes startup hub"
[✓ OK] "Netflix produces new series"
[✓ OK] "Mark Zuckerberg visits Paris"
[✓ OK] "Samsung releases new phone"
[✓ OK] "India wins cricket match"
[✓ OK] "Toyota expands factory in Japan"
[✓ OK] "Angela Merkel meets leaders in Brussels"
[✓ OK] "OpenAI releases new model"
[✓ OK] "Toronto hosts sports event"
[✓ OK] "Bill Gates invests in health tech"

── 22 sentences | 0 warnings


In [33]:
url = "https://www.fdic.gov/bank/individual/failed/banklist.html"
banks = pd.read_html(url)

In [35]:
banks

[                                            Bank Name                City  \
 0             Community Bank and Trust - West Georgia            LaGrange   
 1                   Metropolitan Capital Bank & Trust             Chicago   
 2                        The Santa Anna National Bank          Santa Anna   
 3                                Pulaski Savings Bank             Chicago   
 4                  The First National Bank of Lindsay             Lindsay   
 5               Republic First Bank dba Republic Bank        Philadelphia   
 6                                       Citizens Bank            Sac City   
 7                            Heartland Tri-State Bank             Elkhart   
 8                                 First Republic Bank       San Francisco   
 9                                      Signature Bank            New York   
 10                                Silicon Valley Bank         Santa Clara   
 11                                  Almena State Bank          